# Assignment 10: Image Classification with Convolutional Neural Networks

## Follow These Steps Before Submitting
Once you are finished, ensure to complete the following steps.

1.  Restart your kernel by clicking **'Runtime' > 'Restart session and run all'**.

2.  Fix any errors which result from this.

3.  Repeat steps 1. and 2. until your notebook runs without errors.

4.  Submit your completed notebook to OWL by the deadline.

# Dataset

For this assignment, you will work with the [EuroSAT](https://github.com/phelber/EuroSAT) dataset, a collection of 27,000 georeferenced satellite images from the Sentinel-2 satellite. Each image is 64x64 pixels in RGB and belongs to one of 10 land use / land cover classes: AnnualCrop, Forest, HerbaceousVegetation, Highway, Industrial, Pasture, PermanentCrop, Residential, River, and SeaLake. The dataset is built into `torchvision` and will be downloaded automatically.

In [ ]:
!pip install torchcam --no-deps

In [ ]:
# Imports
import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from torchcam.methods import SmoothGradCAMpp
from torchcam.utils import overlay_mask
from torchvision.transforms.functional import to_pil_image

In [ ]:
# Check if GPU (CUDA) is available, else use CPU

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


# Part 1: Dataset Setup and Exploration

## Question 1.1: Load and prepare the data

1. Define two transform pipelines. The training pipeline should resize images to 224x224, apply a random horizontal flip with 50% probability, apply a random rotation of up to 15 degrees, convert to tensor, and normalize with ImageNet statistics (mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225]). The test pipeline should apply only resizing, tensor conversion, and the same normalization.

2. Download the EuroSAT dataset using `torchvision.datasets.EuroSAT` with `download=True`. Print the class names and total number of images.

3. Use 50% of the dataset to reduce computation time. Split this subset into 80% training and 20% test. Use `torch.manual_seed(2026)` and `torch.randperm` to generate the random indices. Create `Subset` objects with the appropriate transforms applied.

4. Create DataLoaders with a batch size of 32. Shuffle the training loader but not the test loader.

5. Print the number of training and test samples.

In [ ]:
# Training transforms with augmentation
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Test transforms (no augmentation)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Download EuroSAT
base_ds = torchvision.datasets.EuroSAT(root='./data', download=True)
class_names = base_ds.classes
print('classes:', class_names)
print('total images:', len(base_ds))

# Use 50% of data, split 80/20 into train and test
torch.manual_seed(2026)
all_idx = torch.randperm(len(base_ds))
sub_n = len(base_ds) // 2
sub_idx = all_idx[:sub_n]

n_train = int(0.8 * sub_n)
train_idx = sub_idx[:n_train]
test_idx = sub_idx[n_train:]

# Create dataset objects with the right transforms
train_ds = torchvision.datasets.EuroSAT(root='./data', download=False, transform=train_transform)
test_ds = torchvision.datasets.EuroSAT(root='./data', download=False, transform=test_transform)

train_subset = Subset(train_ds, train_idx.tolist())
test_subset = Subset(test_ds, test_idx.tolist())

# DataLoaders
train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=32, shuffle=False)

print('train samples:', len(train_subset))
print('test samples:', len(test_subset))


## Question 1.2: Visualize the dataset

Display one sample image from each of the 10 classes in a 2x5 grid. Use a version of the dataset without normalization (resize and convert to tensor only) so the colors appear natural. Label each image with its class name.

In [ ]:
# Dataset without normalization so colors look right
viz_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])
viz_ds = torchvision.datasets.EuroSAT(root='./data', download=False, transform=viz_transform)

# Grab one example per class
seen = set()
examples = []
for img, label in viz_ds:
    if label not in seen:
        seen.add(label)
        examples.append((img, label))
    if len(seen) == 10:
        break

# Plot 2x5 grid
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()
for i, (img, label) in enumerate(examples):
    ax = axes[i]
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(viz_ds.classes[label])
    ax.axis('off')
plt.tight_layout()
plt.show()


## Question 1.3

Examine the sample images above. Which two classes do you think will be most difficult for a model to distinguish? Explain your reasoning based on visual similarities.

**Written answer:**

forest vs herbaceousvegetation. also river vs sealake (both can look like water + shoreline).


# Part 2: Transfer Learning

## Question 2.1: Set up VGG16

1. Load a pre-trained VGG16 model using the default ImageNet weights.
2. Freeze all parameters in the convolutional feature extractor.
3. Unfreeze the last convolutional layer in the feature extractor (the last 4 modules in `features`).
4. Replace the final layer of the classifier to output 10 classes instead of 1000.
5. Move the model to the GPU.

In [ ]:
# VGG16 with pre-trained weights
weights = models.VGG16_Weights.DEFAULT
vgg16 = models.vgg16(weights=weights)

# Freeze conv layers, then unfreeze the last block
for p in vgg16.features.parameters():
    p.requires_grad = False
for m in list(vgg16.features.children())[-4:]:
    for p in m.parameters():
        p.requires_grad = True

# New classifier head for 10 classes
in_features = vgg16.classifier[-1].in_features
vgg16.classifier[-1] = nn.Linear(in_features, 10)
vgg16 = vgg16.to(device)

print('vgg16 ready')


## Question 2.2: Set up ResNet18

1. Load a pre-trained ResNet18 model using the default ImageNet weights.
2. Freeze all parameters in the model.
3. Unfreeze `layer4` (the last residual block).
4. Replace the fully connected layer (`fc`) to output 10 classes.
5. Move the model to the GPU.

In [ ]:
# ResNet18 with pre-trained weights
weights = models.ResNet18_Weights.DEFAULT
resnet18 = models.resnet18(weights=weights)

# Freeze everything, then unfreeze layer4
for p in resnet18.parameters():
    p.requires_grad = False
for p in resnet18.layer4.parameters():
    p.requires_grad = True

# Replace FC layer for 10 classes
in_features = resnet18.fc.in_features
resnet18.fc = nn.Linear(in_features, 10)
resnet18 = resnet18.to(device)

print('resnet18 ready')


## Question 2.3

Explain the concept of transfer learning. Why is it useful when working with a relatively small satellite image dataset like EuroSAT, and how does freezing and unfreezing layers fit into this approach?

**Written answer:**

transfer learning = start from a model pretrained on a big dataset, then fine-tune it on our smaller dataset. it helps because we don’t have enough data to learn good low-level features from scratch. freezing keeps most pretrained features fixed; unfreezing a few top layers lets the model adapt to eurosat.


# Part 3: Training and Evaluation

## Question 3.1: Train both models

1. Define a training function that takes a model, a DataLoader, a loss function, an optimizer, and the number of epochs. For each epoch, iterate over all batches: move images and labels to the GPU, zero the gradients, compute the forward pass, calculate the loss, backpropagate, and update the weights. Print the average loss per epoch.

2. Use Cross-Entropy Loss and the Adam optimizer with a learning rate of 0.001. When creating the optimizer, pass only the parameters that require gradients.

3. Train VGG16 for 5 epochs, then train ResNet18 for 5 epochs.

In [ ]:
# Training function

def train_model(model, loader, criterion, optimizer, epochs=5):
    model.train()
    for ep in range(epochs):
        running_loss = 0.0
        n = 0
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * imgs.size(0)
            n += imgs.size(0)
        print(f'epoch {ep+1}/{epochs} loss: {running_loss/n:.4f}')


In [ ]:
# Loss and optimizers
criterion = nn.CrossEntropyLoss()

vgg_opt = optim.Adam([p for p in vgg16.parameters() if p.requires_grad], lr=0.001)
res_opt = optim.Adam([p for p in resnet18.parameters() if p.requires_grad], lr=0.001)

# Train VGG16
train_model(vgg16, train_loader, criterion, vgg_opt, epochs=5)

# Train ResNet18
train_model(resnet18, train_loader, criterion, res_opt, epochs=5)


## Question 3.2: Evaluate both models

1. Define an evaluation function that puts the model in evaluation mode, iterates over the test DataLoader with gradients disabled, collects all predictions and true labels, and returns the accuracy along with the prediction and label lists.

2. Evaluate both VGG16 and ResNet18 on the test set. Print the test accuracy for each model.

In [ ]:
# Evaluation function

def eval_model(model, loader):
    model.eval()
    y_true = []
    y_pred = []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            out = model(imgs)
            preds = torch.argmax(out, dim=1)
            y_true.extend(labels.cpu().numpy().tolist())
            y_pred.extend(preds.cpu().numpy().tolist())
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    acc = (y_true == y_pred).mean()
    return acc, y_pred, y_true

# Evaluate both
vgg_acc, vgg_pred, y_true = eval_model(vgg16, test_loader)
res_acc, res_pred, _ = eval_model(resnet18, test_loader)
print(f'vgg16 test acc: {vgg_acc:.4f}')
print(f'resnet18 test acc: {res_acc:.4f}')


## Question 3.3: Confusion matrices

Plot the confusion matrices for both models side by side using `sklearn.metrics.confusion_matrix` and `ConfusionMatrixDisplay`. Use the class names as labels and rotate the x-axis tick labels by 45 degrees for readability.

In [ ]:
# Side-by-side confusion matrices
cm_vgg = confusion_matrix(y_true, vgg_pred, labels=list(range(10)))
cm_res = confusion_matrix(y_true, res_pred, labels=list(range(10)))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
ConfusionMatrixDisplay(cm_vgg, display_labels=class_names).plot(ax=axes[0], colorbar=False)
axes[0].set_title('vgg16')
axes[0].tick_params(axis='x', rotation=45)

ConfusionMatrixDisplay(cm_res, display_labels=class_names).plot(ax=axes[1], colorbar=False)
axes[1].set_title('resnet18')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


## Question 3.4

Compare the training behavior and test performance of VGG16 and ResNet18. Which model converged faster during training? Which achieved higher test accuracy? Based on the confusion matrices, which classes are most commonly confused? Suggest two specific improvements for the weaker model.

**Written answer:**

resnet usually converges faster and gets a bit higher accuracy (deeper but easier to optimize). common confusions: forest vs herbaceous/pasture, and river vs sealake. improvements: train longer + use lr scheduler; unfreeze more layers (or use more aug).


# Part 4: GradCAM Visualization

[SmoothGradCAMpp](https://arxiv.org/abs/1908.01224) is a gradient-based method that produces heatmaps showing which parts of an input image matter most for a given prediction. You used it in Lab 10 with VGG16 on the Intel Image Classification dataset. Here you will apply it to your fine-tuned VGG16 to see what the model looks at when classifying satellite images.

## Question 4.1: GradCAM on correctly classified images

1. Create a `SmoothGradCAMpp` extractor targeting `vgg16.features[28]` (the last convolutional layer).

2. Iterate over the test DataLoader to find 3 correctly classified images from 3 different classes.

3. For each image, run it through the CAM extractor, retrieve the activation map, and display a row of three panels: the original image (denormalized), the raw activation map, and the overlay of the activation map on the original image. Use the `overlay_mask` function from `torchcam.utils`.

Hint: to display the original image without the blue/green tint from ImageNet normalization, reverse the normalization by multiplying by the standard deviation and adding the mean.

In [ ]:
# Helper: reverse ImageNet normalization for display
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def denorm(x):
    x = x.detach().cpu() * std + mean
    return torch.clamp(x, 0, 1)

# Turn on gradients for all params (needed for GradCAM hooks)
for p in vgg16.parameters():
    p.requires_grad = True
vgg16.eval()

# Find 3 correctly classified images from different classes
correct = []
seen = set()
for imgs, labels in test_loader:
    imgs = imgs.to(device)
    labels = labels.to(device)
    out = vgg16(imgs)
    preds = torch.argmax(out, dim=1)
    for i in range(imgs.size(0)):
        y = int(labels[i].item())
        p = int(preds[i].item())
        if y == p and y not in seen:
            seen.add(y)
            correct.append((imgs[i:i+1], y, p))
        if len(correct) == 3:
            break
    if len(correct) == 3:
        break

# Set up CAM extractor on last conv layer and visualize
cam = SmoothGradCAMpp(vgg16, target_layer=vgg16.features[28])

for x, y, p in correct:
    out = vgg16(x)
    act = cam(y, out)[0]

    img = denorm(x[0])
    pil_img = to_pil_image(img)
    heat = to_pil_image(act, mode='F')
    over = overlay_mask(pil_img, heat, alpha=0.5)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(pil_img)
    axes[0].set_title(f'orig ({class_names[y]})')
    axes[0].axis('off')

    axes[1].imshow(act.detach().cpu().numpy(), cmap='jet')
    axes[1].set_title('activation')
    axes[1].axis('off')

    axes[2].imshow(over)
    axes[2].set_title('overlay')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

# Clean up hooks so they don't interfere with the next cell
cam.remove_hooks()


## Question 4.2: GradCAM on misclassified images

Repeat the same visualization for 2 misclassified test images. For each image, show the original (with the true label), the activation map, and the overlay (with the predicted label).

In [ ]:
# Find 2 misclassified images
vgg16.eval()
wrong = []
for imgs, labels in test_loader:
    imgs = imgs.to(device)
    labels = labels.to(device)
    out = vgg16(imgs)
    preds = torch.argmax(out, dim=1)
    for i in range(imgs.size(0)):
        y = int(labels[i].item())
        p = int(preds[i].item())
        if y != p:
            wrong.append((imgs[i:i+1], y, p))
        if len(wrong) == 2:
            break
    if len(wrong) == 2:
        break

# Fresh CAM extractor for this cell
cam = SmoothGradCAMpp(vgg16, target_layer=vgg16.features[28])

# Visualize GradCAM for the misclassified examples
for x, y, p in wrong:
    out = vgg16(x)
    act = cam(p, out)[0]

    img = denorm(x[0])
    pil_img = to_pil_image(img)
    heat = to_pil_image(act, mode='F')
    over = overlay_mask(pil_img, heat, alpha=0.5)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(pil_img)
    axes[0].set_title(f'true: {class_names[y]}')
    axes[0].axis('off')

    axes[1].imshow(act.detach().cpu().numpy(), cmap='jet')
    axes[1].set_title('activation')
    axes[1].axis('off')

    axes[2].imshow(over)
    axes[2].set_title(f'pred: {class_names[p]}')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

# Clean up hooks
cam.remove_hooks()


## Question 4.3

Compare the GradCAM visualizations for the correctly and incorrectly classified images. What regions of the image does the model focus on when it makes correct predictions? Do the misclassified examples reveal any patterns in what confuses the model? Relate your observations to the confusion matrix from Question 3.3.

**Written answer:**

the correct ones usually focus on the main texture/region (like big vegetation blocks or water). wrong ones often focus on mixed areas (water+land) or small patches, so it confuses similar classes (like river vs sealake, forest vs herb/pasture). this matches the confusion matrix.


# End of Assignment 10